# 5. Evaluation: From Gut Checks to Automated Judgment

_From a correct answer to a defensible claim_

Purpose of this section: Give participants the tools and the instinct to evaluate system behavior before making any decision about what to change.

We ended the last section with a win. The model, given the right retrieved context, produced a correct answer. That felt good. It should.

But here's the uncomfortable truth: we only knew it was correct because we already had the answer. We looked it up on page 9 of the rulebook, compared it to what the model returned, and nodded. That's not evaluation. That's spot-checking.

Spot-checking is how customers start. It's also where most proofs of concept quietly die. Someone asks ten questions. Seven look right. Three look wrong. And now what? Is the problem retrieval? Chunking? The model? The question itself? Nobody knows, because nobody built a way to find out.

This is the section where we fix that.
We're going to start exactly where the customer starts: asking questions by hand and writing down what we see. Then we're going to feel why that doesn't scale. And then we're going to build something better.

Run the setup code below to prepare the environment for this notebook.


In [1]:
! pip install pysqlite3-binary ipywidgets -q

import os
os.environ["TQDM_DISABLE"] = "1"

import sys 
sys.path.insert(0, "..") 
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base 
import requests 
import json 
from datetime import datetime

__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import chromadb

# connect to the chromadb
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(name="rpg_rules")

from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("ibm-granite/granite-embedding-30m-english")

print(f"✅ Connected to ChromaDB — {collection.count()} chunks loaded")



/opt/app-root/lib64/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


✅ Connected to ChromaDB — 1205 chunks loaded


## 5.1 The Manual Phase: Ask Questions, Write Down What's Wrong

This is how every customer starts. Someone sits in front of the system, asks a few questions, and forms an opinion. That opinion usually sounds like one of these:

* "It kind of works."
* "Some answers are good, some are weird."
* "I don't trust it yet."

We're going to do the same thing, deliberately. But unlike the customer, we're going to be structured about it. We already have two questions from earlier in the lab. Now we're going to expand that to a small set, run each question two ways, without context and with RAG,  and record what we observe.

This isn't sophisticated. It's not meant to be. The point is to do what the customer does, feel why it breaks down, and understand what we'd need to do it better.


In [2]:
eval_questions = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "known_answer": "The Thief may only try once per lock. If they fail, they must wait until they gain another level of experience before trying again.",
        "why": "Control question — we already verified this in Section 4."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "known_answer": "Elves use a d6 for hit dice because of their character class restrictions in Basic Fantasy RPG.",
        "why": "Tests whether the model answers from this game or defaults to D&D."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "known_answer": "No. Only Magic-Users (and in some cases Thieves at higher levels) can use magic-user scrolls.",
        "why": "Requires interpreting class restriction rules — not a single sentence answer."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "known_answer": "Each side rolls 1d6 at the start of each round. The side with the higher roll acts first.",
        "why": "A rule that differs significantly across RPG systems. Tests domain specificity."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "known_answer": "This depends on the character's Charisma score. The Charisma table defines the maximum number of retainers.",
        "why": "Answer requires connecting two pieces of information — Charisma rules and retainer rules."
    },
]

print(f"Evaluation set: {len(eval_questions)} questions loaded")
for q in eval_questions:
    print(f"  [{q['id']}] {q['question']}")

Evaluation set: 5 questions loaded
  [q1] What happens if a Thief fails an Open Locks attempt?
  [q2] Why can't Elves roll higher than a d6 for hit points?
  [q3] Can a Fighter use magic-user scrolls?
  [q4] How is initiative determined in combat?
  [q5] What is the maximum number of retainers a character can hire?


In [3]:
url_chat = f"{endpoint_base}/chat/completions"
model_id = "granite-3-2-8b-instruct"

def ask_without_context(question, model=model_id):
    """Baseline: model answers from its own training data only."""
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"ERROR: {e}"


def ask_with_rag(question, collection, model=model_id, n_results=3):
    """RAG: retrieve relevant chunks, then ask the model."""
    # Retrieve (embed the question manually)
    query_embedding = embed_model.encode(question).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    context_chunks = results["documents"][0]
    context = "\n\n---\n\n".join(context_chunks)

    # Build prompt
    system_msg = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so."
    )
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        answer = response.json()["choices"][0]["message"]["content"]
        return answer, context_chunks
    except Exception as e:
        return f"ERROR: {e}", context_chunks

In [4]:
results = []

print("Running evaluation set...\n")
for q in eval_questions:
    print(f"[{q['id']}] {q['question']}")

    # Baseline (no context)
    baseline_answer = ask_without_context(q["question"])

    # RAG (with retrieved context)
    rag_answer, retrieved_chunks = ask_with_rag(q["question"], collection)

    result = {
        "id": q["id"],
        "question": q["question"],
        "known_answer": q["known_answer"],
        "baseline_answer": baseline_answer,
        "rag_answer": rag_answer,
        "retrieved_chunks": retrieved_chunks,
        "timestamp": datetime.now().isoformat()
    }
    results.append(result)
    print(f"  ✅ Done\n")

print(f"All {len(results)} questions complete.")


Running evaluation set...

[q1] What happens if a Thief fails an Open Locks attempt?
  ✅ Done

[q2] Why can't Elves roll higher than a d6 for hit points?
  ✅ Done

[q3] Can a Fighter use magic-user scrolls?
  ✅ Done

[q4] How is initiative determined in combat?
  ✅ Done

[q5] What is the maximum number of retainers a character can hire?
  ✅ Done

All 5 questions complete.


In [5]:
print("=" * 80)
print("EVALUATION RESULTS: Baseline vs. RAG")
print("=" * 80)

for r in results:
    print(f"\n{'─' * 80}")
    print(f"Q{r['id']}: {r['question']}")
    print(f"{'─' * 80}")
    print(f"\n🧠 Baseline (no context):\n{r['baseline_answer']}")
    print(f"\n📚 RAG (with retrieved context):\n{r['rag_answer']}")
    print(f"\n📎 Retrieved chunks: {len(r['retrieved_chunks'])}")

EVALUATION RESULTS: Baseline vs. RAG

────────────────────────────────────────────────────────────────────────────────
Qq1: What happens if a Thief fails an Open Locks attempt?
────────────────────────────────────────────────────────────────────────────────

🧠 Baseline (no context):
If a Thief fails an Open Locks attempt, they do not make any progress towards opening the lock. The Thief must wait until their next attempt to try again. If the Thief fails the roll by 4 or more, they may trigger a trap or alert nearby enemies.

📚 RAG (with retrieved context):
If a Thief fails an Open Locks attempt, they won't be able to unlock the lock without a proper key. Furthermore, they must wait until they have gained another level of experience before trying again.

📎 Retrieved chunks: 3

────────────────────────────────────────────────────────────────────────────────
Qq2: Why can't Elves roll higher than a d6 for hit points?
─────────────────────────────────────────────────────────────────────────

### 5.1.1 Evaluate Manually

Before we automate anything, we evaluate by hand. This is deliberate. Automated metrics are useful, but they only matter if you first understand what 'good' and 'bad' look like in your domain. No scoring framework can replace the judgment of someone who knows the data.

The following cell builds a simple tabbed interface that shows each test question side by side: the baseline answer (no context) and the RAG answer (with retrieved context). For each one, you score it on a three-point scale: wrong or hallucinated, partial or vague, or correct and grounded. There is also space for notes if you want to capture why you scored something a particular way.

This is the same kind of exercise you would walk a customer through. The goal is not to get perfect scores. The goal is to build a shared understanding of where the system works, where it fails, and whether those failures point to retrieval, data quality, or something deeper.

In [6]:
import ipywidgets as widgets
from IPython.display import display, HTML

scores = {}

def create_evaluation_ui(results):
    tabs = []
    
    for r in results:
        q_id = r["id"]
        
        # Display content
        content = widgets.HTML(value=f"""
        <h3>Q{q_id}: {r['question']}</h3>
        <h4>🧠 Baseline (no context):</h4>
        <p style="background:rgba(128,128,128,0.15); padding:10px; border-radius:5px;">{r['baseline_answer']}</p>
        <h4>📚 RAG (with retrieved context):</h4>
        <p style="background:rgba(76,175,80,0.15); padding:10px; border-radius:5px;">{r['rag_answer']}</p>
        """)
        
        # Scoring dropdowns
        baseline_score = widgets.Dropdown(
            options=[("—", None), ("0 - Wrong/Hallucinated", 0), ("1 - Partial/Vague", 1), ("2 - Correct/Grounded", 2)],
            description="Baseline:",
            style={"description_width": "80px"}
        )
        rag_score = widgets.Dropdown(
            options=[("—", None), ("0 - Wrong/Hallucinated", 0), ("1 - Partial/Vague", 1), ("2 - Correct/Grounded", 2)],
            description="RAG:",
            style={"description_width": "80px"}
        )
        
        notes = widgets.Textarea(
            placeholder="Optional notes — why did you score it this way?",
            layout=widgets.Layout(width="100%", height="60px")
        )
        
        # Save scores on change
        def make_handler(qid, field):
            def handler(change):
                if qid not in scores:
                    scores[qid] = {}
                scores[qid][field] = change["new"]
            return handler
        
        baseline_score.observe(make_handler(q_id, "baseline"), names="value")
        rag_score.observe(make_handler(q_id, "rag"), names="value")
        notes.observe(make_handler(q_id, "notes"), names="value")
        
        tab_content = widgets.VBox([content, baseline_score, rag_score, notes])
        tabs.append(tab_content)
    
    tab_widget = widgets.Tab(children=tabs)
    for i, r in enumerate(results):
        tab_widget.set_title(i, f"Q{r['id']}")
    
    display(tab_widget)

create_evaluation_ui(results)

⬆️ Look at the interactive interface above.

Take your time with this. Look at each answer critically. When the RAG answer is better, ask yourself why. When it is not, ask where the breakdown happened. Was the right chunk retrieved? Was the chunk complete? Did the model ignore relevant context?

These observations become the evidence you use to decide what to do next. Without them, every optimization is a guess.

Once you have scored all the questions, run the next cell to see the results in a summary table. This gives you a simple, defensible comparison: baseline versus RAG, question by question, with the delta between them. The numbers here are not a benchmark. They are a conversation starter. If the delta is consistently positive, RAG is earning its place. If it is flat or negative on certain questions, that tells you something specific about where the system needs attention. This is the kind of artifact you bring into a customer meeting, not to prove the system works, but to show that you know exactly where it does and where it does not.

In [7]:
def show_scores():
    print("EVALUATION SUMMARY")
    print("=" * 50)
    print(f"{'Question':<12} {'Baseline':>10} {'RAG':>10} {'Delta':>10}")
    print("-" * 50)
    
    baseline_total = 0
    rag_total = 0
    
    for q_id, s in sorted(scores.items()):
        b = s.get("baseline", "—")
        r = s.get("rag", "—")
        delta = ""
        if isinstance(b, int) and isinstance(r, int):
            delta = f"+{r - b}" if r >= b else str(r - b)
            baseline_total += b
            rag_total += r
        print(f"Q{q_id:<11} {str(b):>10} {str(r):>10} {delta:>10}")
    
    print("-" * 50)
    print(f"{'Total':<12} {baseline_total:>10} {rag_total:>10} {f'+{rag_total - baseline_total}':>10}")

show_scores()

EVALUATION SUMMARY
Question       Baseline        RAG      Delta
--------------------------------------------------
--------------------------------------------------
Total                 0          0         +0



### 5.1.2 What you'll likely see:

For most questions, the RAG answer is noticeably better. It uses the right terminology, references the right rules, and doesn't default to D&D. That's the improvement we expected from Section 4.
But you'll also likely see something else:
* An answer that's correct but vaguely worded
* An answer where the retrieved chunks were relevant but the model still hedged
* An answer where the wrong chunks got retrieved — and the model confidently answered with the wrong context
* An answer where the model said it couldn't find enough information, even though the answer exists in the document

These are not model failures. These are system behaviors. And each one points to a different layer.
The problem you should be feeling right now is this: you just spent several minutes reading five questions. A real customer document set would generate dozens or hundreds of test questions. Some answers will be clearly right. Some will be clearly wrong. Most will be somewhere in between.

Manual evaluation gives you insight. It does not give you scale.

We'll fix that. But first, let's capture what we learned.


## 5.2 Why Manual Doesn't Scale 
You just spent several minutes reading five answers. You compared them to what you expected. You formed judgments. Some were easy — the answer was clearly right or clearly wrong. Others were harder. The model said something plausible but slightly off, and you had to decide whether that counted.

Now imagine doing that with fifty questions. Or two hundred. Across four models. Every time someone changes the chunking strategy or swaps an embedding model or adds new documents to the corpus.

This is where manual evaluation quietly dies. Not because it's wrong — the judgments you just made were valuable. But because it doesn't survive contact with iteration. Every time the system changes, you'd need to re-read, re-compare, and re-judge. 

The cost isn't just time. It's the consistency of your judgement. Your assessment of "partially correct" at 9 AM after coffee may not match your assessment of "partially correct" at 4 PM after twelve other reviews.

Customers hit this wall faster than you'd expect. They start with enthusiasm — someone on the team volunteers to test the system, asks twenty questions, writes up observations. That works once. It doesn't work the second time, because now they need to compare against the first round. By the third iteration, the volunteer has moved on to other priorities and nobody remembers what "good enough" looked like in round one.

This is the gap between evaluation and evaluability. Evaluation is something a person does once. Evaluability is something the system supports permanently. The first is a judgment call. The second is infrastructure.

We're not going to build a full evaluation framework in this lab. But we are going to take one step that matters: we're going to define what "correct" looks like in a way that a machine can check. Not perfectly. Not comprehensively. But in a repeatable manner.
That's the shift. From "I read it and it seemed right" to "these keywords were present or they weren't." It's crude. It's also the difference between a demo and a system.


## 5.3 Structured Feedback: The Thumbs Up/Down Step 

Before we automate anything, there's an intermediate step that most teams skip. It sits between "I read the answer and it seemed fine" and "we have a full evaluation pipeline." It's simple, and it's powerful precisely because it's simple.

Instead of asking "is this answer good?" (which invites subjectivity, debate, and inconsistency),  we ask a narrower question: "does this answer contain the things we'd expect a correct answer to contain?"

That's keyword scoring. For each question, we define a short list of terms that should appear in a correct response. Not the exact phrasing. Not the full sentence. Just the signals. If someone asks how initiative works in Basic Fantasy and the answer doesn't mention "d6," something is wrong. If someone asks about Open Locks and the answer never says "once per lock" or "another level," the model missed it, regardless of how confident the response sounded.




In [8]:
eval_set = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "expected_keywords": ["wait", "another level", "experience", "once", "lock"],
        "reference_answer": "The Thief must wait until they have gained another level of experience before trying again. It may only be tried once per lock."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "expected_keywords": ["d6", "hit die", "elf", "elves", "magic-user", "combination"],
        "reference_answer": "Elves use a d6 for hit dice because they are a combination Fighter/Magic-User class, and their hit die reflects the Magic-User side."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "expected_keywords": ["no", "cannot", "magic-user", "arcane", "spell caster"],
        "reference_answer": "No. Only spell casters can use magic-user scrolls."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "expected_keywords": ["d6", "each side", "roll", "beginning", "round"],
        "reference_answer": "Each side rolls 1d6 at the beginning of each round. The side with the highest roll acts first."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "expected_keywords": ["charisma", "bonus", "4", "retainer"],
        "reference_answer": "The number of retainers is based on the character's Charisma bonus. A character with average Charisma can hire up to 4."
    },
]

print(f"✅ Evaluation set defined — {len(eval_set)} questions with expected answers")

✅ Evaluation set defined — 5 questions with expected answers


In [9]:
def keyword_score(answer, expected_keywords):
    """Score based on how many expected keywords appear in the answer."""
    answer_lower = answer.lower()
    hits = [kw for kw in expected_keywords if kw.lower() in answer_lower]
    misses = [kw for kw in expected_keywords if kw.lower() not in answer_lower]
    score = len(hits) / len(expected_keywords) if expected_keywords else 0
    return {
        "score": round(score, 2),
        "hits": hits,
        "misses": misses
    }

def run_structured_eval(eval_set, results):
    """Run keyword scoring against both baseline and RAG answers."""
    scored = []
    
    for ev in eval_set:
        # Find the matching result
        match = next((r for r in results if r["id"] == ev["id"]), None)
        if not match:
            print(f"⚠️  No result found for {ev['id']}, skipping")
            continue
        
        baseline_eval = keyword_score(match["baseline_answer"], ev["expected_keywords"])
        rag_eval = keyword_score(match["rag_answer"], ev["expected_keywords"])
        
        scored.append({
            "id": ev["id"],
            "question": ev["question"],
            "reference": ev["reference_answer"],
            "baseline_answer": match["baseline_answer"],
            "rag_answer": match["rag_answer"],
            "baseline_score": baseline_eval,
            "rag_score": rag_eval,
        })
    
    return scored

scored_results = run_structured_eval(eval_set, results)
print(f"✅ Structured evaluation complete — {len(scored_results)} questions scored")

✅ Structured evaluation complete — 5 questions scored


In [10]:
display(HTML("""
<div style="background:rgba(255,152,0,0.1); padding:12px; border-radius:8px; border-left:4px solid #FF9800;">
<h3>5.3 — Structured Evaluation: Keyword Coverage</h3>
<p>Instead of eyeballing, we now check whether answers contain the key terms 
we'd expect from a correct response. This isn't perfect — but it's <strong>repeatable</strong> and <strong>defensible</strong>.</p>
</div>
"""))

for s in scored_results:
    b = s["baseline_score"]
    r = s["rag_score"]
    delta = r["score"] - b["score"]
    indicator = "📈" if delta > 0 else "📉" if delta < 0 else "➡️"
    
    print(f"\n{'═' * 70}")
    print(f"Q{s['id']}: {s['question']}")
    print(f"{'═' * 70}")
    
    print(f"\n📖 Reference answer:")
    print(f"   {s['reference']}")
    
    print(f"\n🧠 Baseline — score: {b['score']:.0%}")
    print(f"   ✓ hits:   {', '.join(b['hits']) or '(none)'}")
    print(f"   ✗ misses: {', '.join(b['misses']) or '(none)'}")
    
    print(f"\n📚 RAG — score: {r['score']:.0%}")
    print(f"   ✓ hits:   {', '.join(r['hits']) or '(none)'}")
    print(f"   ✗ misses: {', '.join(r['misses']) or '(none)'}")
    
    print(f"\n{indicator} Delta: {delta:+.0%}")


══════════════════════════════════════════════════════════════════════
Qq1: What happens if a Thief fails an Open Locks attempt?
══════════════════════════════════════════════════════════════════════

📖 Reference answer:
   The Thief must wait until they have gained another level of experience before trying again. It may only be tried once per lock.

🧠 Baseline — score: 40%
   ✓ hits:   wait, lock
   ✗ misses: another level, experience, once

📚 RAG — score: 80%
   ✓ hits:   wait, another level, experience, lock
   ✗ misses: once

📈 Delta: +40%

══════════════════════════════════════════════════════════════════════
Qq2: Why can't Elves roll higher than a d6 for hit points?
══════════════════════════════════════════════════════════════════════

📖 Reference answer:
   Elves use a d6 for hit dice because they are a combination Fighter/Magic-User class, and their hit die reflects the Magic-User side.

🧠 Baseline — score: 33%
   ✓ hits:   d6, elves
   ✗ misses: hit die, elf, magic-user, com

And, in case the message hasn’t landed yet, here’s a summary.

In [11]:
print("STRUCTURED EVALUATION SUMMARY")
print("=" * 55)
print(f"{'Question':<10} {'Baseline':>12} {'RAG':>12} {'Delta':>12}")
print("-" * 55)

b_total = 0
r_total = 0

for s in scored_results:
    b = s["baseline_score"]["score"]
    r = s["rag_score"]["score"]
    delta = r - b
    b_total += b
    r_total += r
    indicator = "📈" if delta > 0 else "📉" if delta < 0 else "➡️"
    print(f"Q{s['id']:<9} {b:>11.0%} {r:>11.0%} {delta:>+11.0%} {indicator}")

n = len(scored_results)
print("-" * 55)
print(f"{'Average':<10} {b_total/n:>11.0%} {r_total/n:>11.0%} {(r_total-b_total)/n:>+11.0%}")

print(f"\n{'─' * 55}")
print("Key takeaway:")
if r_total > b_total:
    print("  RAG improves keyword coverage — retrieval is adding real value.")
    print("  Next: Are the remaining misses a retrieval problem or a model problem?")
elif r_total == b_total:
    print("  No measurable improvement. Check if retrieval is finding the right chunks.")
else:
    print("  RAG is underperforming baseline. Retrieved context may be introducing noise.")



STRUCTURED EVALUATION SUMMARY
Question       Baseline          RAG        Delta
-------------------------------------------------------
Qq1                40%         80%        +40% 📈
Qq2                33%         33%         +0% ➡️
Qq3                60%         60%         +0% ➡️
Qq4                20%         40%        +20% 📈
Qq5                50%        100%        +50% 📈
-------------------------------------------------------
Average            41%         63%        +22%

───────────────────────────────────────────────────────
Key takeaway:
  RAG improves keyword coverage — retrieval is adding real value.
  Next: Are the remaining misses a retrieval problem or a model problem?


The numbers are in, and they tell a clear story. RAG is adding real value. Keyword coverage improved across the board — not because we changed the model, not because we tweaked a prompt, but because we gave the model access to the right information at the right time. That's the data pipeline doing its job.

But look at what didn't improve. Some questions still have missing keywords after RAG. Some scores barely moved. And that raises the question you should now be asking every time you see a gap between what the system produced and what you expected.

### 5.3.1 Where is this failing?
There are really only two possibilities. Either the retrieval layer didn't find the right chunks — which means the model never had a chance — or the retrieval layer did its job and the model still couldn't produce the right answer. Those are fundamentally different problems with fundamentally different fixes. One is a data pipeline problem. The other is a model capability problem. Treating them the same is how teams end up fine-tuning a model to compensate for bad chunking.

So before we touch anything else, we need to pull one more variable apart. We've been running every question against a single model. What happens if we run the same evaluation (same questions, same retrieved context, same keyword scoring) across every model available on our endpoint?

If all the models improve roughly the same amount with RAG, the story is straightforward: the win is in the retrieval, and model choice is secondary. But if one model jumps dramatically while others stay flat, or if one model consistently misses keywords that others catch, that tells us something different. It tells us the model matters for this domain and **that's a data point we'd need before any conversation about customization**.

This is what Section 5.4 is for. 

* Same eval set. 
* Same scoring. 
* Every model. 
* One table.



## 5.4 Automated Evaluation Across Different Models

Up to this point, we've been running every question through a single model. That was deliberate as we needed to isolate variables. First we tested whether retrieval improved answers. It did. 
Then we tested whether we could measure that improvement in a repeatable manner. We can.

Now we change one variable: the model itself.

This is the test that most teams skip. They pick a model early — often based on a blog post, a vendor recommendation, or whatever was already deployed — and never revisit the decision. 

Every subsequent conversation about quality becomes a conversation about that one model. "The model struggles with our terminology." "The model hallucinates on edge cases." "We need to fine-tune the model."

Maybe. Or maybe a different model handles the domain better out of the box, and the entire fine-tuning conversation was premature.
We have four models on our endpoint. We have five questions with keyword scoring already defined. The infrastructure is in place. So instead of guessing which model is "best," we're going to measure it.

If you're curious, this is a good time to experiment. The MaaS dashboard lets you subscribe to additional models beyond the four we set up in Section 2. If you want to see how a different model handles these same questions, add it to your subscription, regenerate your API key to include it, and the evaluation code will pick it up automatically. The more models you test, the stronger your evidence becomes and the harder it gets to argue that model choice alone is the problem. 

Just keep in mind that every model you add multiplies the number of API calls: more models means more time and more tokens consumed. Four models is enough to see the pattern. Eight models makes the pattern undeniable. Sixteen models means you're probably stalling on making a decision.

The next cell fetches every model available on the MaaS endpoint and filters out the ones that aren't designed for chat: embedding models and guardrail models can't answer questions, so we exclude them. 

What's left is our evaluation lineup:


In [12]:
models_url = f"{endpoint_base}/models"
headers = {"Authorization": f"Bearer {key}"}
response = requests.get(models_url, headers=headers)
available_models = [m["id"] for m in response.json()["data"]]

# Filter out embedding models — they can't do chat completions
skip = ["nomic-embed", "Llama-Guard"]
chat_models = [m for m in available_models if not any(s.lower() in m.lower() for s in skip)]

print(f"Models available for evaluation: {len(chat_models)}")
for m in chat_models:
    print(f"  • {m}")

Models available for evaluation: 4
  • granite-4-0-h-tiny
  • qwen3-14b
  • microsoft-phi-4
  • granite-3-2-8b-instruct


### 5.4.1 Run every question against every model: baseline and RAG

Now we run the same evaluation we've been doing, but for every model in our lineup. Each question gets asked twice per model: once with no context (baseline) and once with RAG. Same questions, same retrieved chunks, same keyword scoring. The only variable that changes is the model doing the answering.

This is the kind of comparison that would take hours to do manually. You'd open a chat interface, switch models, re-ask the questions, copy-paste answers into a spreadsheet, squint at the differences. We're going to do it in a single cell.

The code below loops through every chat-capable model on the endpoint, runs all five evaluation questions both ways, and stores the results. It will take a couple of minutes depending on how many models you're subscribed to. Watch the output as it runs — you'll start noticing patterns before the scoring even begins.


In [13]:
multi_model_results = {}

for model in chat_models:
    print(f"\n{'='*60}")
    print(f"🤖 Model: {model}")
    print(f"{'='*60}")
    
    model_results = []
    for q in eval_questions:
        print(f"  [{q['id']}] {q['question'][:50]}...", end=" ")
        
        baseline = ask_without_context(q["question"], model=model)
        rag_answer, chunks = ask_with_rag(q["question"], collection, model=model)
        
        model_results.append({
            "id": q["id"],
            "question": q["question"],
            "baseline_answer": baseline,
            "rag_answer": rag_answer,
        })
        print("✅")
    
    multi_model_results[model] = model_results

print(f"\n✅ All done — {len(chat_models)} models × {len(eval_questions)} questions")


🤖 Model: granite-4-0-h-tiny
  [q1] What happens if a Thief fails an Open Locks attemp... ✅
  [q2] Why can't Elves roll higher than a d6 for hit poin... ✅
  [q3] Can a Fighter use magic-user scrolls?... ✅
  [q4] How is initiative determined in combat?... ✅
  [q5] What is the maximum number of retainers a characte... ✅

🤖 Model: qwen3-14b
  [q1] What happens if a Thief fails an Open Locks attemp... ✅
  [q2] Why can't Elves roll higher than a d6 for hit poin... ✅
  [q3] Can a Fighter use magic-user scrolls?... ✅
  [q4] How is initiative determined in combat?... ✅
  [q5] What is the maximum number of retainers a characte... ✅

🤖 Model: microsoft-phi-4
  [q1] What happens if a Thief fails an Open Locks attemp... ✅
  [q2] Why can't Elves roll higher than a d6 for hit poin... ✅
  [q3] Can a Fighter use magic-user scrolls?... ✅
  [q4] How is initiative determined in combat?... ✅
  [q5] What is the maximum number of retainers a characte... ✅

🤖 Model: granite-3-2-8b-instruct
  [q1] What happen

### 5.4.2 Score every model's answers

We have raw answers from every model. Now we apply the same keyword scoring we built in Section 5.3 — but across the full matrix. Every model, every question, baseline and RAG, scored against the same expected keywords.

This is where the evaluation framework starts earning its keep. We defined those keywords once. Now we're reusing them across four models without re-reading a single answer. That's the difference between a one-time check and a repeatable process.



In [14]:
multi_model_scores = {}

for model, model_results in multi_model_results.items():
    scored = []
    for ev in eval_set:
        match = next((r for r in model_results if r["id"] == ev["id"]), None)
        if not match:
            continue
        b = keyword_score(match["baseline_answer"], ev["expected_keywords"])
        r = keyword_score(match["rag_answer"], ev["expected_keywords"])
        scored.append({"id": ev["id"], "baseline": b["score"], "rag": r["score"]})
    multi_model_scores[model] = scored

### 5.4.3 The comparison table

This is the payoff for everything we've built in Section 5. One table. Every model. Every question. Baseline versus RAG. Color-coded so the patterns are visible at a glance.

In [15]:
from IPython.display import display, HTML

# Build an HTML table for the cross-model comparison
model_names = list(multi_model_scores.keys())
short_names = [m.replace("granite-3-2-8b-instruct", "Granite 3.2 8B")
                .replace("granite-4-0-h-tiny", "Granite 4.0 Tiny")
                .replace("microsoft-phi-4", "Phi-4")
                .replace("qwen3-14b", "Qwen3 14B")
               for m in model_names]

html = """
<style>
.eval-table { border-collapse: collapse; font-family: monospace; font-size: 13px; }
.eval-table th, .eval-table td { padding: 8px 14px; text-align: center; border: 1px solid #444; }
.eval-table th { background: #2a2a2a; color: #fff; }
.eval-table tr:nth-child(even) { background: #f8f8f8; }
.eval-table .label { text-align: left; font-weight: bold; }
.eval-table .avg-row { background: #e8e8e8; font-weight: bold; }
.improve { color: #2e7d32; } 
.decline { color: #c62828; }
.neutral { color: #666; }
</style>
<table class="eval-table">
<tr><th>Question</th>"""

for name in short_names:
    html += f"<th>{name}</th>"
html += "</tr>"

# Data rows
for i, ev in enumerate(eval_set):
    html += f"<tr><td class='label'>{ev['id'].upper()}</td>"
    for model in model_names:
        s = multi_model_scores[model][i]
        b, r = s["baseline"], s["rag"]
        delta = r - b
        css = "improve" if delta > 0 else "decline" if delta < 0 else "neutral"
        arrow = "→"
        html += f"<td><span class='neutral'>{b:.0%}</span> {arrow} <span class='{css}'>{r:.0%}</span></td>"
    html += "</tr>"

# Average row
html += "<tr class='avg-row'><td class='label'>AVG</td>"
for model in model_names:
    scores = multi_model_scores[model]
    avg_b = sum(s["baseline"] for s in scores) / len(scores)
    avg_r = sum(s["rag"] for s in scores) / len(scores)
    delta = avg_r - avg_b
    css = "improve" if delta > 0 else "decline" if delta < 0 else "neutral"
    html += f"<td><span class='neutral'>{avg_b:.0%}</span> → <span class='{css}'>{avg_r:.0%}</span></td>"
html += "</tr></table>"

display(HTML(html))

Question,Granite 4.0 Tiny,Qwen3 14B,Phi-4,Granite 3.2 8B
Q1,20% → 80%,40% → 80%,40% → 100%,40% → 80%
Q2,33% → 33%,50% → 33%,33% → 33%,33% → 33%
Q3,40% → 20%,40% → 60%,60% → 60%,60% → 60%
Q4,20% → 40%,20% → 60%,20% → 40%,20% → 40%
Q5,25% → 100%,50% → 100%,25% → 100%,50% → 100%
AVG,28% → 55%,40% → 67%,36% → 67%,41% → 63%


Read it horizontally first: how did each question perform across models? Then read it vertically: how did each model perform across questions? The story that emerges from those two readings is what you'd bring into a customer conversation. Not "this model is better" but "here's where the data pipeline helped, here's where it didn't, and here's what that tells us about what to do next."

## 5.5 What Evaluation Tells You (and What It Doesn't)

Look at the table you just produced. There's a lot of information in it, and it's tempting to draw big conclusions. Before you do, let's be honest about what this evaluation can and cannot tell us.

What it tells you is real. RAG improved keyword coverage for most questions across most models. That's not an opinion, it's a measurement. But look at the exceptions. Q2 stayed flat or declined across every model. That means the retrieval layer didn't find the right chunk, or the chunk didn't contain what the model needed. That's not a model problem. That's a data pipeline problem, and it tells you exactly where to investigate next. And notice Q3 for Granite 4.0 Tiny, where adding context actually made the answer worse. RAG is not always additive, especially with smaller models. More context can overwhelm a model that doesn't have the capacity to sort signal from noise. These aren't failures of the approach. They're diagnostic information. The table is doing exactly what it should: showing you where the system works and where it doesn't.

It also tells you where the gaps are. Look at Q2. Every model flatlined or regressed. That's a pattern, and patterns have causes. Did the retriever pull the wrong chunks? Did the model ignore what it was given? Did our keywords fail to capture what a correct answer actually looks like? Each of those is a different problem with a different fix. But the fact that every model behaved the same way on Q2 tells you the issue is upstream of the model. Start there. Now look at Q5, where every model jumped dramatically. That's what it looks like when the right chunk lands cleanly in context. The system worked as designed. The contrast between Q2 and Q5 is the whole story of this lab in two rows.

What it doesn't tell you is equally important. Keyword scoring can't evaluate reasoning. An answer could contain every expected keyword and still be incoherent. It could assemble the right terms into the wrong conclusion. It could quote the source material verbatim without actually answering the question. Those failures require a human to catch, and in a production system, they require a more sophisticated evaluation approach than what we built in this section.

It also doesn't tell you whether these results generalize. Five questions is enough to see patterns. It's not enough to make claims about production readiness. A real evaluation set for this domain would need dozens of questions covering different rule categories, edge cases, and question types. Building that set is work, but it's work that pays for itself every time the system changes, because you can re-run it in minutes instead of re-reading answers for hours.

And finally, it doesn't tell you what to do. Evaluation produces evidence. Decisions require judgment. The table might show that one model outperforms the others by a few percentage points. Is that enough to choose it? Depends on cost, latency, licensing, and a dozen other factors the table can't capture. The table narrows the conversation. It doesn't end it.

Think of what we built as a floor, not a ceiling. It's the minimum evaluation infrastructure that lets you make evidence-based decisions about this system. In a real engagement, you'd build on it with more questions, more scoring methods, maybe an LLM-as-judge approach where one model evaluates another's answers. But even this simple version changes the conversation from "I think it's working" to "here's what's working and here's what isn't."

That's the shift that matters.


### 5.5.1 A Quick Note about LLM-as-Judge (LLMaJ)
Keyword scoring has a hard ceiling. It can tell you whether the right words showed up, but not whether the answer was coherent, well-reasoned, or actually responsive to what the user asked. Catching those failures requires something closer to human judgment, and that's where LLM-as-Judge comes in.

The idea is simple: instead of a human reviewing each answer, a second language model does it. The generator produces a response, and the judge evaluates it against a criterion like overall quality, faithfulness to the source material, or relevance to the question. Unlike human review, it scales. You can run a judge over hundreds of questions in the time it would take a person to read ten.

There are real limitations. The judge is still an LLM, which means it can be confidently wrong, and its scoring behavior depends heavily on how you prompt it. The judge model should also be at least as capable as the generator it's evaluating, otherwise it can't reliably recognize quality it couldn't produce itself.

### 5.5.2 Where RAGAS Fits

The keyword scoring you built here was never meant to be the final answer. It was meant to give you enough intuition about your data to know what a better metric should measure. That is what RAGAS, an open source evaluation framework designed specifically for RAG systems, provides. 

Where keyword overlap asks "did the answer contain the right words," RAGAS asks better questions. Faithfulness measures whether every claim in the generated answer can be traced back to the retrieved context. Answer Relevancy measures whether the answer actually addresses what was asked. Context Precision and Context Recall evaluate whether the retriever surfaced the right material. Factual Correctness compares the answer against a known reference.

For a RAG system with any compliance requirement, Faithfulness is the metric to reach for first. A score below 0.8 is a conversation worth having before any model changes are considered.

RAGAS is available across the Red Hat portfolio: as a Python package, as part of TrustyAI, as part of EvalHub, and as part of Llama Stack. 

The full list of metrics is at https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/ 




## 5.6 Framing This for Customers

Everything you just did in this section, the manual review, the keyword scoring, the cross-model comparison, maps directly to conversations you'll have in the field. The question is how to translate what you learned into language that lands with a customer who hasn't been sitting in this lab.

When a customer says "we need better answers," the instinct is to talk about models. Bigger models, fine-tuned models, different models. What you now know is that the first question isn't about the model at all. It's: "How are you measuring better?" If they can't answer that, neither can you.

And any change you make, to retrieval, to chunking, to the model itself, becomes unjustifiable, because there's no before and after to compare.
So the first thing you bring into a customer conversation isn't a recommendation. It's a framework. You show them that evaluation is a prerequisite, not an afterthought. You help them define what "correct" looks like for their domain. You build a small question set together, ten questions, maybe twenty, with expected answers their subject matter expert agrees on. That alone is often the most productive meeting in the entire engagement, because it forces the customer to articulate standards they've never written down.

From there, the conversation becomes structured. "We tested five models against your evaluation set. Here's what we found. RAG improved every model by this much. These questions still fail consistently. Here's where we think the gap is." That's a conversation an engineering lead can take to their leadership. It's a conversation a compliance team can engage with. It's the opposite of "we tried it and it seemed pretty good."

And when the customer inevitably asks "should we fine-tune?"

You now have the answer that earns trust instead of burning it. Not "yes" or "no," but "here's what the evaluation shows. Here's what's failing and why. Here's what we'd try next before we consider changing the model." That positions you as someone who solves problems methodically, not someone who sells the most expensive option.

The evaluation table you built in this section is a prototype. It took minutes. But the discipline it represents: measure before you change, compare before 
you escalate, defend before you recommend. That's what separates a proof of concept that dies after the demo from a system that makes it to production.

And production is exactly where we need to point this next.

Everything we've built so far, the ingestion pipeline, the chunking strategy, the vector store, the evaluation framework, the cross-model comparison, was built against a single document. One rulebook. One PDF. That was the right call. It allowed us to isolate variables, understand behavior, and build confidence one layer at a time.

**But no customer has one document.**

They have twelve. Or two hundred. Or twelve thousand. They have policy manuals that reference procedure guides that reference training materials that contradict last year's version of themselves. They have documents written by different people in different decades using different terminology for the same concepts.

The question we need to answer now isn't whether the pipeline works. We proved that. The question is whether it survives.